# Creación del Datamart Analítico

Este notebook construye el datamart en esquema estrella a partir de las tablas operacionales
(`creditos`, `amortizacion`, `juicios`) cargadas por el ETL.

### Estrategia

1. **Vista materializada `mv_creditos_mensuales`**: encapsula el JOIN de 3 tablas + agregación
   por mes-bloque + cálculo de tasas y crisis_flag. Es la **fuente de verdad única** para
   entrenamiento, predicción y BI.
2. **Dimensiones**: `dim_tiempo`, `dim_riesgo`, `dim_sector`, `dim_sucursal` (upsert idempotente).
3. **Tabla de hechos**: `fact_creditos_mensual` poblada desde la MV con joins a dimensiones.
4. **Validación**: verificación de integridad y consistencia.

### Entradas
- Tablas: `creditos`, `amortizacion`, `juicios` en PostgreSQL.

### Salidas
- Vista materializada: `mv_creditos_mensuales`
- Dimensiones: `dim_tiempo`, `dim_riesgo`, `dim_sector`, `dim_sucursal`
- Tabla de hechos: `fact_creditos_mensual`

## 1. Configuración

In [22]:
%load_ext autoreload
%autoreload 2

import logging
import os
import pandas as pd
from datetime import datetime
from sqlalchemy import create_engine, text

logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s - %(levelname)s - %(message)s",
    force=True,
)
logger = logging.getLogger(__name__)

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [23]:
DB_CONFIG = {
    "host": "localhost",
    "port": "5432",
    "database": "postgres_db",
    "user": "postgres_usr",
    "password": "admin123",
}

connection_string = (
    f"postgresql://{DB_CONFIG['user']}:{DB_CONFIG['password']}"
    f"@{DB_CONFIG['host']}:{DB_CONFIG['port']}/{DB_CONFIG['database']}"
)
engine = create_engine(connection_string)

print("=" * 80)
print("CREACIÓN DEL DATAMART ANALÍTICO")
print("=" * 80)
print(f"Base de datos: {DB_CONFIG['database']}")
print(f"Fecha: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
print("=" * 80)

CREACIÓN DEL DATAMART ANALÍTICO
Base de datos: postgres_db
Fecha: 2026-07-06 16:32:12


## 2. Crear tablas de dimensiones y hechos (DDL)

Se crea el esquema estrella una sola vez. Las tablas se mantienen si ya existen.

In [24]:
ddl_dim_tiempo = """
CREATE TABLE IF NOT EXISTS dim_tiempo (
    id_tiempo SERIAL PRIMARY KEY,
    mes DATE NOT NULL UNIQUE,
    anio INTEGER NOT NULL,
    trimestre INTEGER NOT NULL,
    mes_del_anio INTEGER NOT NULL,
    nombre_mes VARCHAR(20) NOT NULL
);
"""

ddl_dim_riesgo = """
CREATE TABLE IF NOT EXISTS dim_riesgo (
    id_riesgo SERIAL PRIMARY KEY,
    codigo_riesgo VARCHAR(50) NOT NULL UNIQUE,
    descripcion VARCHAR(100)
);
"""

ddl_dim_sector = """
CREATE TABLE IF NOT EXISTS dim_sector (
    id_sector SERIAL PRIMARY KEY,
    codigo_sector VARCHAR(100) NOT NULL UNIQUE,
    descripcion VARCHAR(100)
);
"""

ddl_dim_sucursal = """
DROP TABLE IF EXISTS dim_sucursal CASCADE;
CREATE TABLE dim_sucursal (
    id_sucursal SERIAL PRIMARY KEY,
    codigo_sucursal INTEGER NOT NULL UNIQUE,
    codigo_provincia INTEGER DEFAULT 0
);
"""

ddl_fact_creditos = """
DROP TABLE IF EXISTS fact_creditos_mensual CASCADE;
CREATE TABLE fact_creditos_mensual (
    id_tiempo INTEGER NOT NULL,
    id_riesgo INTEGER NOT NULL,
    id_sector INTEGER NOT NULL,
    id_sucursal INTEGER NOT NULL,
    num_creditos INTEGER NOT NULL,
    monto_total NUMERIC(18,2),
    monto_promedio NUMERIC(18,2),
    dias_mora_promedio NUMERIC(10,2),
    num_moras_promedio NUMERIC(10,2),
    tasa_mora_90 NUMERIC(8,2),
    tasa_judicial NUMERIC(8,2),
    tasa_cierre NUMERIC(8,2),
    total_gestion_cobro NUMERIC(18,2),
    total_costo_judicial NUMERIC(18,2),
    tasa_interes_promedio NUMERIC(8,2),
    saldo_promedio NUMERIC(18,2),
    creditos_cerrados INTEGER,
    num_clientes_unicos INTEGER,
    creditos_por_cliente NUMERIC(8,2),
    plazo_promedio NUMERIC(8,2),
    desviacion_montos NUMERIC(18,2),
    coef_variacion_montos NUMERIC(15,2),
    antiguedad_promedio_meses NUMERIC(15,2),
    tasa_crecimiento_creditos NUMERIC(15,2),
    tasa_crecimiento_monto NUMERIC(15,2),
    crisis_flag INTEGER DEFAULT 0,
    bloque_id VARCHAR(200) NOT NULL,
    PRIMARY KEY (id_tiempo, id_riesgo, id_sector, id_sucursal)
);
"""

with engine.connect() as conn:
    for nombre, ddl in [
        ("dim_tiempo", ddl_dim_tiempo),
        ("dim_riesgo", ddl_dim_riesgo),
        ("dim_sector", ddl_dim_sector),
        ("dim_sucursal", ddl_dim_sucursal),
        ("fact_creditos_mensual", ddl_fact_creditos),
    ]:
        conn.execute(text(ddl))
        conn.commit()
        print(f"  OK: {nombre}")

print("\nEsquema estrella creado/verificado.")

  OK: dim_tiempo
  OK: dim_riesgo
  OK: dim_sector
  OK: dim_sucursal
  OK: fact_creditos_mensual

Esquema estrella creado/verificado.


## 3. Crear vista materializada `mv_creditos_mensuales`

La vista materializada encapsula la lógica completa de:
- JOIN de las 3 tablas operacionales
- Agregación por mes-bloque (riesgo × sector × sucursal)
- Cálculo de tasas derivadas (mora_90, judicial, cierre, etc.)
- Cálculo del `crisis_flag` con 11 reglas de scoring
- Generación del `bloque_id`

Esta es la **fuente de verdad única** que reemplaza las queries duplicadas en EDA, train y predict.

In [25]:
sql_drop_mv = "DROP MATERIALIZED VIEW IF EXISTS mv_creditos_mensuales;"

sql_create_mv = """
CREATE MATERIALIZED VIEW mv_creditos_mensuales AS

WITH datos_crudos AS (
    SELECT
        c.numero_credito,
        c.codigo_act_financiera,
        c.codigo_producto,
        c.fecha_credito,
        c.cant_soli,
        c.num_cuotas,
        c.tasa_interes,
        c.tot_dias_mora,
        c.tot_num_moras,
        c.estado_cred,
        c.judicial,
        c.capital_porpagar,
        c.codigo_sucursal,
        c.porc_pig,
        c.costo_judicial,
        c.notificaciones,
        c.gestion_cobro,
        c.monto_real,
        c.saldo_capital,
        a.capitalcal,
        a.interescal,
        a.moracal,
        a.rubroscal,
        a.totalcal,
        j.valor_demanda,
        j.capital_recuperado
    FROM creditos c
    JOIN amortizacion a ON c.numero_credito = a.numero_credito
    LEFT JOIN juicios j ON c.numero_credito = j.numero_credito
    WHERE c.fecha_credito >= '2005-01-01'
),

datos_mensuales AS (
    SELECT
        DATE_TRUNC('month', fecha_credito)::DATE AS mes,
        COALESCE(codigo_act_financiera::TEXT, 'SIN_RIESGO') AS riesgo,
        COALESCE(codigo_producto::TEXT, 'SIN_SECTOR') AS sector,
        codigo_sucursal,
        COUNT(*) AS num_creditos,
        SUM(COALESCE(cant_soli, 0)) AS monto_total,
        AVG(COALESCE(cant_soli, 0)) AS monto_promedio,
        AVG(COALESCE(tot_dias_mora, 0)) AS dias_mora_promedio,
        AVG(COALESCE(tot_num_moras, 0)) AS num_moras_promedio,
        COUNT(CASE WHEN tot_dias_mora > 90 THEN 1 END) AS creditos_mora_90,
        COUNT(CASE WHEN judicial = 'S' THEN 1 END) AS creditos_judiciales,
        SUM(COALESCE(gestion_cobro::NUMERIC, 0)) AS total_gestion_cobro,
        SUM(COALESCE(costo_judicial, 0)) AS total_costo_judicial,
        AVG(COALESCE(tasa_interes, 0)) AS tasa_interes_promedio,
        AVG(COALESCE(saldo_capital, 0)) AS saldo_promedio,
        COUNT(CASE WHEN estado_cred IN ('C', 'L') THEN 1 END) AS creditos_cerrados,
        COUNT(DISTINCT numero_credito) AS num_clientes_unicos,
        EXTRACT(MONTH FROM DATE_TRUNC('month', fecha_credito))::INTEGER AS mes_del_anio,
        AVG(COALESCE(num_cuotas, 0)) AS plazo_promedio,
        STDDEV(COALESCE(cant_soli, 0)) AS desviacion_montos,
        AVG(EXTRACT(MONTH FROM AGE(CURRENT_DATE, fecha_credito))) AS antiguedad_promedio_meses
    FROM datos_crudos
    GROUP BY
        DATE_TRUNC('month', fecha_credito),
        codigo_act_financiera,
        codigo_producto,
        codigo_sucursal
),

datos_con_lag AS (
    SELECT
        *,
        LAG(num_creditos, 1) OVER (
            PARTITION BY riesgo, sector ORDER BY mes
        ) AS num_creditos_mes_anterior,
        LAG(monto_total, 1) OVER (
            PARTITION BY riesgo, sector ORDER BY mes
        ) AS monto_mes_anterior
    FROM datos_mensuales
),

datos_enriquecidos AS (
    SELECT
        mes,
        riesgo,
        sector,
        codigo_sucursal,
        num_creditos,
        monto_total,
        monto_promedio,
        dias_mora_promedio,
        num_moras_promedio,
        ROUND((creditos_mora_90::NUMERIC / NULLIF(num_creditos, 0)) * 100, 2) AS tasa_mora_90,
        ROUND((creditos_judiciales::NUMERIC / NULLIF(num_creditos, 0)) * 100, 2) AS tasa_judicial,
        ROUND((creditos_cerrados::NUMERIC / NULLIF(num_creditos, 0)) * 100, 2) AS tasa_cierre,
        total_gestion_cobro,
        total_costo_judicial,
        tasa_interes_promedio,
        saldo_promedio,
        creditos_cerrados,
        num_clientes_unicos,
        ROUND(num_creditos::NUMERIC / NULLIF(num_clientes_unicos, 0), 2) AS creditos_por_cliente,
        mes_del_anio,
        ROUND(plazo_promedio::NUMERIC, 2) AS plazo_promedio,
        ROUND(COALESCE(desviacion_montos, 0)::NUMERIC, 2) AS desviacion_montos,
        ROUND((COALESCE(desviacion_montos, 0)::NUMERIC / NULLIF(monto_promedio, 0)) * 100, 2) AS coef_variacion_montos,
        ROUND(antiguedad_promedio_meses::NUMERIC, 2) AS antiguedad_promedio_meses,
        num_creditos_mes_anterior,
        ROUND(
            ((num_creditos::NUMERIC - COALESCE(num_creditos_mes_anterior, num_creditos))
             / NULLIF(num_creditos_mes_anterior, 0)) * 100, 2
        ) AS tasa_crecimiento_creditos,
        monto_mes_anterior,
        ROUND(
            ((monto_total::NUMERIC - COALESCE(monto_mes_anterior, monto_total))
             / NULLIF(monto_mes_anterior, 0)) * 100, 2
        ) AS tasa_crecimiento_monto
    FROM datos_con_lag
    WHERE num_creditos >= 10
),

datos_con_crisis AS (
    SELECT *,
        CASE
            WHEN (
                CASE WHEN tasa_mora_90 > 15 THEN 3 ELSE 0 END +
                CASE WHEN tasa_judicial > 5 THEN 2 ELSE 0 END +
                CASE WHEN dias_mora_promedio > 45 THEN 2 ELSE 0 END +
                CASE WHEN total_gestion_cobro > monto_total * 0.08 THEN 1 ELSE 0 END +
                CASE WHEN creditos_cerrados::NUMERIC / NULLIF(num_creditos, 0) > 0.3 THEN 1 ELSE 0 END +
                CASE WHEN num_creditos < 50 AND tasa_mora_90 > 20 THEN 2 ELSE 0 END +
                CASE WHEN creditos_por_cliente > 3 THEN 1 ELSE 0 END +
                CASE WHEN tasa_crecimiento_creditos < -20 THEN 1 ELSE 0 END +
                CASE WHEN coef_variacion_montos > 100 THEN 1 ELSE 0 END +
                CASE WHEN plazo_promedio > 36 AND tasa_mora_90 > 10 THEN 1 ELSE 0 END +
                CASE WHEN antiguedad_promedio_meses > 60 AND tasa_judicial > 3 THEN 1 ELSE 0 END
            ) >= 4 THEN 1
            ELSE 0
        END AS crisis_flag
    FROM datos_enriquecidos
)

SELECT
    mes,
    riesgo,
    sector,
    codigo_sucursal,
    num_creditos,
    monto_total,
    monto_promedio,
    dias_mora_promedio,
    num_moras_promedio,
    tasa_mora_90,
    tasa_judicial,
    tasa_cierre,
    total_gestion_cobro,
    total_costo_judicial,
    tasa_interes_promedio,
    saldo_promedio,
    creditos_cerrados,
    num_clientes_unicos,
    creditos_por_cliente,
    plazo_promedio,
    desviacion_montos,
    coef_variacion_montos,
    antiguedad_promedio_meses,
    tasa_crecimiento_creditos,
    tasa_crecimiento_monto,
    crisis_flag,
    riesgo || '_' || sector || '_' || codigo_sucursal::TEXT AS bloque_id
FROM datos_con_crisis;
"""

sql_create_idx_mv = """
CREATE UNIQUE INDEX IF NOT EXISTS idx_mv_creditos_mensuales_pk
    ON mv_creditos_mensuales(mes, riesgo, sector, codigo_sucursal);
"""

with engine.connect() as conn:
    logger.info("Eliminando vista materializada anterior (si existe)...")
    conn.execute(text(sql_drop_mv))
    conn.commit()

    logger.info("Creando vista materializada mv_creditos_mensuales...")
    conn.execute(text(sql_create_mv))
    conn.commit()

    logger.info("Creando índice único para REFRESH CONCURRENTLY...")
    conn.execute(text(sql_create_idx_mv))
    conn.commit()

print("Vista materializada mv_creditos_mensuales creada.")

2026-07-06 16:32:12,894 - INFO - Eliminando vista materializada anterior (si existe)...
2026-07-06 16:32:12,903 - INFO - Creando vista materializada mv_creditos_mensuales...
2026-07-06 16:32:18,915 - INFO - Creando índice único para REFRESH CONCURRENTLY...


Vista materializada mv_creditos_mensuales creada.


### Verificar la vista materializada

In [26]:
df_mv = pd.read_sql("SELECT COUNT(*) AS total FROM mv_creditos_mensuales", engine)
print(f"Registros en mv_creditos_mensuales: {df_mv['total'].iloc[0]:,}")

df_sample = pd.read_sql(
    "SELECT * FROM mv_creditos_mensuales ORDER BY mes LIMIT 5",
    engine
)
print("\nMuestra:")
df_sample

Registros en mv_creditos_mensuales: 12,109

Muestra:


,mes,riesgo,sector,codigo_sucursal,num_creditos,monto_total,monto_promedio,dias_mora_promedio,num_moras_promedio,tasa_mora_90,...,num_clientes_unicos,creditos_por_cliente,plazo_promedio,desviacion_montos,coef_variacion_montos,antiguedad_promedio_meses,tasa_crecimiento_creditos,tasa_crecimiento_monto,crisis_flag,bloque_id
0,2019-01-01,2,13,2,28,1008000.0,36000.000000,0.0,0.0,0.0,...,1,28.0,28.00,0.00,0.0,5.00,-53.33,-32.80,0,2_13_2
1,2019-01-01,2,13,29,60,1500000.0,25000.000000,0.0,0.0,0.0,...,1,60.0,60.00,0.00,0.0,5.00,-61.54,-91.78,0,2_13_29
2,2019-01-01,2,13,49,156,18240000.0,116923.076923,0.0,0.0,0.0,...,3,52.0,54.46,37409.94,32.0,5.23,NaN,NaN,0,2_13_49
3,2019-01-01,2,2,2,66,153000.0,2318.181818,0.0,0.0,0.0,...,2,33.0,39.82,1121.94,48.4,5.00,120.00,183.33,0,2_2_2
4,2019-01-01,2,2,3,48,240000.0,5000.000000,0.0,0.0,0.0,...,1,48.0,48.00,0.00,0.0,5.00,-38.46,33.33,0,2_2_3


## 4. Poblar dimensiones

Se extraen los valores únicos de la vista materializada y se insertan de forma idempotente
(sin duplicados) usando `ON CONFLICT DO NOTHING`.

In [27]:
sql_dim_tiempo = """
INSERT INTO dim_tiempo (mes, anio, trimestre, mes_del_anio, nombre_mes)
SELECT DISTINCT
    mes,
    EXTRACT(YEAR FROM mes)::INTEGER,
    EXTRACT(QUARTER FROM mes)::INTEGER,
    EXTRACT(MONTH FROM mes)::INTEGER,
    TO_CHAR(mes, 'TMMonth')
FROM mv_creditos_mensuales
ON CONFLICT (mes) DO NOTHING;
"""

sql_dim_riesgo = """
INSERT INTO dim_riesgo (codigo_riesgo, descripcion)
SELECT DISTINCT riesgo, riesgo
FROM mv_creditos_mensuales
ON CONFLICT (codigo_riesgo) DO NOTHING;
"""

sql_dim_sector = """
INSERT INTO dim_sector (codigo_sector, descripcion)
SELECT DISTINCT sector, sector
FROM mv_creditos_mensuales
ON CONFLICT (codigo_sector) DO NOTHING;
"""

sql_dim_sucursal = """
INSERT INTO dim_sucursal (codigo_sucursal, codigo_provincia)
SELECT DISTINCT codigo_sucursal, 0
FROM mv_creditos_mensuales
ON CONFLICT (codigo_sucursal) DO NOTHING;
"""

with engine.connect() as conn:
    for nombre, sql in [
        ("dim_tiempo", sql_dim_tiempo),
        ("dim_riesgo", sql_dim_riesgo),
        ("dim_sector", sql_dim_sector),
        ("dim_sucursal", sql_dim_sucursal),
    ]:
        result = conn.execute(text(sql))
        conn.commit()
        print(f"  {nombre}: {result.rowcount} filas insertadas")

print("\nDimensiones pobladas.")

  dim_tiempo: 36 filas insertadas
  dim_riesgo: 1 filas insertadas
  dim_sector: 16 filas insertadas
  dim_sucursal: 64 filas insertadas

Dimensiones pobladas.


In [28]:
for tabla in ["dim_tiempo", "dim_riesgo", "dim_sector", "dim_sucursal"]:
    df = pd.read_sql(f"SELECT COUNT(*) AS total FROM {tabla}", engine)
    print(f"  {tabla}: {df['total'].iloc[0]:,} registros")

  dim_tiempo: 36 registros
  dim_riesgo: 1 registros
  dim_sector: 16 registros
  dim_sucursal: 64 registros


## 5. Poblar tabla de hechos `fact_creditos_mensual`

Se hace JOIN de la vista materializada con las 4 dimensiones para resolver las FKs,
y se inserta con upsert para ser idempotente.

In [29]:
sql_fact_upsert = """
INSERT INTO fact_creditos_mensual (
    id_tiempo, id_riesgo, id_sector, id_sucursal,
    num_creditos, monto_total, monto_promedio,
    dias_mora_promedio, num_moras_promedio,
    tasa_mora_90, tasa_judicial, tasa_cierre,
    total_gestion_cobro, total_costo_judicial,
    tasa_interes_promedio, saldo_promedio,
    creditos_cerrados, num_clientes_unicos,
    creditos_por_cliente, plazo_promedio,
    desviacion_montos, coef_variacion_montos,
    antiguedad_promedio_meses,
    tasa_crecimiento_creditos, tasa_crecimiento_monto,
    crisis_flag, bloque_id
)
SELECT
    dt.id_tiempo,
    dr.id_riesgo,
    ds.id_sector,
    dsu.id_sucursal,
    mv.num_creditos,
    mv.monto_total,
    mv.monto_promedio,
    mv.dias_mora_promedio,
    mv.num_moras_promedio,
    mv.tasa_mora_90,
    mv.tasa_judicial,
    mv.tasa_cierre,
    mv.total_gestion_cobro,
    mv.total_costo_judicial,
    mv.tasa_interes_promedio,
    mv.saldo_promedio,
    mv.creditos_cerrados,
    mv.num_clientes_unicos,
    mv.creditos_por_cliente,
    mv.plazo_promedio,
    mv.desviacion_montos,
    mv.coef_variacion_montos,
    mv.antiguedad_promedio_meses,
    mv.tasa_crecimiento_creditos,
    mv.tasa_crecimiento_monto,
    mv.crisis_flag,
    mv.bloque_id
FROM mv_creditos_mensuales mv
JOIN dim_tiempo dt ON dt.mes = mv.mes
JOIN dim_riesgo dr ON dr.codigo_riesgo = mv.riesgo
JOIN dim_sector ds ON ds.codigo_sector = mv.sector
JOIN dim_sucursal dsu ON dsu.codigo_sucursal = mv.codigo_sucursal
ON CONFLICT (id_tiempo, id_riesgo, id_sector, id_sucursal) DO UPDATE SET
    num_creditos               = EXCLUDED.num_creditos,
    monto_total                = EXCLUDED.monto_total,
    monto_promedio             = EXCLUDED.monto_promedio,
    dias_mora_promedio         = EXCLUDED.dias_mora_promedio,
    num_moras_promedio         = EXCLUDED.num_moras_promedio,
    tasa_mora_90               = EXCLUDED.tasa_mora_90,
    tasa_judicial              = EXCLUDED.tasa_judicial,
    tasa_cierre                = EXCLUDED.tasa_cierre,
    total_gestion_cobro        = EXCLUDED.total_gestion_cobro,
    total_costo_judicial       = EXCLUDED.total_costo_judicial,
    tasa_interes_promedio      = EXCLUDED.tasa_interes_promedio,
    saldo_promedio             = EXCLUDED.saldo_promedio,
    creditos_cerrados          = EXCLUDED.creditos_cerrados,
    num_clientes_unicos        = EXCLUDED.num_clientes_unicos,
    creditos_por_cliente       = EXCLUDED.creditos_por_cliente,
    plazo_promedio             = EXCLUDED.plazo_promedio,
    desviacion_montos          = EXCLUDED.desviacion_montos,
    coef_variacion_montos      = EXCLUDED.coef_variacion_montos,
    antiguedad_promedio_meses  = EXCLUDED.antiguedad_promedio_meses,
    tasa_crecimiento_creditos  = EXCLUDED.tasa_crecimiento_creditos,
    tasa_crecimiento_monto     = EXCLUDED.tasa_crecimiento_monto,
    crisis_flag                = EXCLUDED.crisis_flag,
    bloque_id                  = EXCLUDED.bloque_id;
"""

with engine.connect() as conn:
    logger.info("Poblando fact_creditos_mensual...")
    result = conn.execute(text(sql_fact_upsert))
    conn.commit()
    print(f"fact_creditos_mensual: {result.rowcount} filas insertadas/actualizadas")

2026-07-06 16:32:19,120 - INFO - Poblando fact_creditos_mensual...


fact_creditos_mensual: 12109 filas insertadas/actualizadas


## 6. Validación

In [30]:
print("=" * 60)
print("VALIDACIÓN DEL DATAMART")
print("=" * 60)

# 1. Conteo por tabla
print("\n1) Conteo de registros:")
for tabla in ["mv_creditos_mensuales", "dim_tiempo", "dim_riesgo",
              "dim_sector", "dim_sucursal", "fact_creditos_mensual"]:
    df = pd.read_sql(f"SELECT COUNT(*) AS total FROM {tabla}", engine)
    print(f"   {tabla}: {df['total'].iloc[0]:,}")

# 2. Distribución de crisis_flag
print("\n2) Distribución de crisis_flag:")
df_crisis = pd.read_sql(
    "SELECT crisis_flag, COUNT(*) AS cantidad FROM fact_creditos_mensual GROUP BY crisis_flag ORDER BY crisis_flag",
    engine
)
print(df_crisis.to_string(index=False))

# 3. Verificar FKs nulas
print("\n3) Verificación de FKs nulas (debe ser 0):")
df_nulls = pd.read_sql(
    "SELECT COUNT(*) AS nulos FROM fact_creditos_mensual WHERE id_tiempo IS NULL OR id_riesgo IS NULL OR id_sector IS NULL OR id_sucursal IS NULL",
    engine
)
print(f"   Filas con FK nula: {df_nulls['nulos'].iloc[0]}")

# 4. Verificar duplicados
print("\n4) Verificación de duplicados (debe ser 0):")
df_dups = pd.read_sql(
    "SELECT COUNT(*) AS duplicados FROM (SELECT id_tiempo, id_riesgo, id_sector, id_sucursal FROM fact_creditos_mensual GROUP BY id_tiempo, id_riesgo, id_sector, id_sucursal HAVING COUNT(*) > 1) t",
    engine
)
print(f"   Filas duplicadas: {df_dups['duplicados'].iloc[0]}")

# 5. Rango de fechas
print("\n5) Rango de fechas en dim_tiempo:")
df_rango = pd.read_sql(
    "SELECT MIN(mes) AS inicio, MAX(mes) AS fin FROM dim_tiempo",
    engine
)
print(f"   Desde: {df_rango['inicio'].iloc[0]}  Hasta: {df_rango['fin'].iloc[0]}")

print("\n" + "=" * 60)

VALIDACIÓN DEL DATAMART

1) Conteo de registros:
   mv_creditos_mensuales: 12,109
   dim_tiempo: 36
   dim_riesgo: 1
   dim_sector: 16
   dim_sucursal: 64
   fact_creditos_mensual: 12,109

2) Distribución de crisis_flag:
 crisis_flag  cantidad
           0     10190
           1      1919

3) Verificación de FKs nulas (debe ser 0):
   Filas con FK nula: 0

4) Verificación de duplicados (debe ser 0):
   Filas duplicadas: 0

5) Rango de fechas en dim_tiempo:
   Desde: 2019-01-01  Hasta: 2021-12-01



In [31]:
print("Muestra de fact_creditos_mensual (primeras 10 filas):")
df_fact_sample = pd.read_sql(
    """
    SELECT f.*, dt.mes, dr.codigo_riesgo, ds.codigo_sector
    FROM fact_creditos_mensual f
    JOIN dim_tiempo dt ON f.id_tiempo = dt.id_tiempo
    JOIN dim_riesgo dr ON f.id_riesgo = dr.id_riesgo
    JOIN dim_sector ds ON f.id_sector = ds.id_sector
    ORDER BY dt.mes
    LIMIT 10
    """,
    engine
)
df_fact_sample

Muestra de fact_creditos_mensual (primeras 10 filas):


,id_tiempo,id_riesgo,id_sector,id_sucursal,num_creditos,monto_total,monto_promedio,dias_mora_promedio,num_moras_promedio,tasa_mora_90,...,desviacion_montos,coef_variacion_montos,antiguedad_promedio_meses,tasa_crecimiento_creditos,tasa_crecimiento_monto,crisis_flag,bloque_id,mes,codigo_riesgo,codigo_sector
0,29,1,1,33,14,350000.00,25000.00,0.0,0.0,0.0,...,5188.75,20.75,5.00,133.33,45.83,0,2_47_15,2019-01-01,2,47
1,29,1,3,1,60,223523.40,3725.39,0.0,0.0,0.0,...,0.00,0.00,5.00,62.16,24.17,0,2_34_49,2019-01-01,2,34
2,29,1,3,5,85,35296.47,415.25,0.0,0.0,0.0,...,355.44,85.60,5.00,63.46,-75.66,0,2_34_7,2019-01-01,2,34
3,29,1,3,6,64,11579.55,180.93,0.0,0.0,0.0,...,99.97,55.25,5.28,-26.44,-90.88,0,2_34_19,2019-01-01,2,34
4,29,1,3,7,71,51309.09,722.66,0.0,0.0,0.0,...,587.51,81.30,5.17,NaN,NaN,0,2_34_32,2019-01-01,2,34
5,29,1,3,8,694,1174331.00,1692.12,0.0,0.0,0.0,...,2194.77,129.71,5.02,837.84,1845.92,0,2_34_3,2019-01-01,2,34
6,29,1,3,11,751,1074105.91,1430.23,0.0,0.0,0.0,...,1899.79,132.83,5.05,725.27,4754.57,0,2_34_14,2019-01-01,2,34
7,29,1,3,15,68,157483.66,2315.94,0.0,0.0,0.0,...,2490.98,107.56,5.00,-4.23,206.93,0,2_34_33,2019-01-01,2,34
8,29,1,3,16,37,180020.00,4865.41,0.0,0.0,0.0,...,818.71,16.83,5.00,311.11,19461.01,0,2_34_47,2019-01-01,2,34
9,29,1,3,18,87,23671.07,272.08,0.0,0.0,0.0,...,224.77,82.61,5.00,27.94,-84.97,0,2_34_35,2019-01-01,2,34


## 7. Consultas de ejemplo para Superset

Estas consultas sirven como base para los dashboards.

In [32]:
print("Top 10 bloques con mayor tasa_judicial promedio:")
df_top = pd.read_sql(
    """
    SELECT bloque_id,
           ROUND(AVG(tasa_judicial), 2) AS tasa_judicial_prom,
           ROUND(AVG(tasa_mora_90), 2) AS tasa_mora_90_prom,
           SUM(num_creditos) AS total_creditos,
           SUM(crisis_flag) AS meses_crisis
    FROM fact_creditos_mensual
    GROUP BY bloque_id
    ORDER BY tasa_judicial_prom DESC
    LIMIT 10
    """,
    engine
)
df_top

Top 10 bloques con mayor tasa_judicial promedio:


,bloque_id,tasa_judicial_prom,tasa_mora_90_prom,total_creditos,meses_crisis
0,2_35_41,100.00,0.0,85,3
1,2_43_56,67.60,0.0,1161,3
2,2_35_56,50.00,0.0,108,1
3,2_13_8,50.00,0.0,52,1
4,2_32_19,50.00,0.0,300,1
5,2_44_56,42.25,0.0,1650,9
6,2_34_43,40.30,0.0,428,3
7,2_44_41,37.79,0.0,4125,14
8,2_44_49,34.72,0.0,38262,26
9,2_13_49,33.33,0.0,448,2


In [33]:
print("Evolución mensual de tasa_mora_90 por sector:")
df_evol = pd.read_sql(
    """
    SELECT dt.mes, ds.codigo_sector,
           ROUND(AVG(f.tasa_mora_90), 2) AS tasa_mora_90_prom,
           SUM(f.num_creditos) AS num_creditos
    FROM fact_creditos_mensual f
    JOIN dim_tiempo dt ON f.id_tiempo = dt.id_tiempo
    JOIN dim_sector ds ON f.id_sector = ds.id_sector
    GROUP BY dt.mes, ds.codigo_sector
    ORDER BY dt.mes, ds.codigo_sector
    LIMIT 30
    """,
    engine
)
df_evol

Evolución mensual de tasa_mora_90 por sector:


,mes,codigo_sector,tasa_mora_90_prom,num_creditos
0,2019-01-01,13,0.0,244
1,2019-01-01,2,0.0,1717
2,2019-01-01,32,0.0,2208
3,2019-01-01,34,0.0,6521
4,2019-01-01,35,0.0,3656
5,2019-01-01,41,0.0,3839
6,2019-01-01,42,0.0,3373
7,2019-01-01,43,0.0,41994
8,2019-01-01,44,0.0,109388
9,2019-01-01,45,0.0,83


## 8. Refresco incremental

Para refrescar el datamart con datos nuevos, ejecutar estas celdas.
La vista materializada se recrea (o se usa `REFRESH` si ya tiene índice único).
Las dimensiones y la fact table se actualizan con upsert.

In [34]:
# Descomentar para refrescar:
# with engine.connect() as conn:
#     logger.info("Refrescando vista materializada...")
#     conn.execute(text("REFRESH MATERIALIZED VIEW CONCURRENTLY mv_creditos_mensuales"))
#     conn.commit()
#     print("MV refrescada.")
#
#     for sql in [sql_dim_tiempo, sql_dim_riesgo, sql_dim_sector, sql_dim_sucursal]:
#         conn.execute(text(sql))
#     conn.commit()
#     print("Dimensiones actualizadas.")
#
#     result = conn.execute(text(sql_fact_upsert))
#     conn.commit()
#     print(f"Fact table: {result.rowcount} filas actualizadas.")

![icon](../../DocumentosBase/yachayCuadrado.jpg)<br/>***<omar.velez@yachaytech.edu.ec>***<br/>*julio 2026*